# Grad-CAM Explainability

This notebook runs signature-specific patch attribution, starting from already-trained checkpoints.

For each patch, slide, and mutational signature, the script computes:
- **`grad_score`**: `Σ_d h[i,d] · ∂y_k/∂h[i,d]` — signed patch contribution to signature k
- **`attention_weight`**: MIL attention weight (shared across all signatures)

Gradients are computed w.r.t. `h` (output of `instance_encoder`), not the raw input.

The notebook covers two datasets:
- **TCGA** (5-fold CV) — uses `best.pt` per fold + `val_attention_patches.csv` as a slide filter
- **CPTAC** (external test) — uses `hist2sig.pt` (model trained on full TCGA train+val), no CSV filter

---

## Required Files

**TCGA — per fold (1–5):**

| File | Location |
|---|---|
| `best.pt` | `results/fold_{N}/` |
| `val_attention_patches.csv` | same directory as the checkpoint |
| features `*.pt` | `/leonardo_work/CNHPC_2116672/features_complete/` |

**CPTAC:**

| File | Location |
|---|---|
| `hist2sig.pt` | `checkpoints/hist2sig.pt` |
| features `*.pt` | `/leonardo_work/CNHPC_2116672/Hist2Sig/features/` |

CPTAC requires no CSV filter: the feature directory already contains only CPTAC slides.

---

## Minimum Dependencies

The repository must be cloned; the script imports from `src/train_activities_nb.py` and `utils/sampling_utils.py`.
The only extra Python dependencies (beyond torch and numpy) are `pandas` and `pyarrow`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "pandas", "pyarrow", "tqdm"], check=True)

## 1. Path Configuration

Edit only this cell.

In [ ]:
from pathlib import Path

REPO_DIR        = Path("/leonardo/home/userexternal/cpancott/Hist2Sig")
RESULTS_DIR     = REPO_DIR / "results"
CHECKPOINTS_DIR = REPO_DIR / "checkpoints"
SCRIPT      = REPO_DIR / "grad_cam_signatures" / "export_signature_patch_attribution.py"

# TCGA
TCGA_FEATURES_DIR = Path("/leonardo_work/CNHPC_2116672/features_complete")
TCGA_OUT_BASE     = Path("/leonardo_work/CNHPC_2116672/grad_cam_signatures_tcga")
TCGA_FOLDS        = [1, 2, 3, 4, 5]

# CPTAC
CPTAC_FEATURES_DIR = Path("/leonardo_work/CNHPC_2116672/Hist2Sig/features")
CPTAC_OUT_BASE     = Path("/leonardo_work/CNHPC_2116672/grad_cam_signatures_cptac")

# Sampling parameters (shared between TCGA and CPTAC)
BAGSIZE       = 10000
SAMPLING      = "energy_topk"
ENERGY_MODE   = "l2_sq"
ENERGY_TAU    = 1.0
SAMPLING_SEED = 42

print("Script:         ", SCRIPT)
print("TCGA features:  ", TCGA_FEATURES_DIR)
print("CPTAC features: ", CPTAC_FEATURES_DIR)

---
# Part A — TCGA (5-Fold Cross-Validation)

## A1. TCGA Prerequisites Check

In [ ]:
ok = True

if not SCRIPT.exists():
    print(f"[MISSING] script: {SCRIPT}"); ok = False

if not TCGA_FEATURES_DIR.exists():
    print(f"[MISSING] TCGA features: {TCGA_FEATURES_DIR}"); ok = False
else:
    n = len(list(TCGA_FEATURES_DIR.glob("*.pt")))
    print(f"[OK] TCGA features: {n} .pt files")

tcga_fold_configs = []
for fold in TCGA_FOLDS:
    export_dir = RESULTS_DIR / f"fold_{fold}"
    ckpt       = export_dir / "best.pt"
    val_csv    = export_dir / "val_attention_patches.csv"
    out_file   = TCGA_OUT_BASE / f"val_gradcam_fold{fold}.parquet"

    ckpt_ok = ckpt.exists()
    val_ok  = val_csv.exists()
    status  = "[OK]" if (ckpt_ok and val_ok) else "[MISSING]"
    print(f"{status} fold{fold}")
    if not ckpt_ok:
        print(f"         {ckpt}"); ok = False
    if not val_ok:
        print(f"         {val_csv}"); ok = False

    tcga_fold_configs.append(dict(fold=fold, ckpt=ckpt, val_csv=val_csv, out_file=out_file))

print()
print("TCGA OK" if ok else "Fix missing files before proceeding.")

## A2. Run TCGA Attribution

In [ ]:
import subprocess, sys

TCGA_OUT_BASE.mkdir(parents=True, exist_ok=True)

def run_gradcam(ckpt, features_dir, out_file, val_csv=None, label=""):
    if out_file.exists():
        print(f"{label}: already exists, skipping → {out_file}")
        return 0
    cmd = [
        sys.executable, str(SCRIPT),
        "--features-dir",  str(features_dir),
        "--ckpt",          str(ckpt),
        "--out-parquet",   str(out_file),
        "--bagsize",       str(BAGSIZE),
        "--sampling",      SAMPLING,
        "--energy-mode",   ENERGY_MODE,
        "--energy-tau",    str(ENERGY_TAU),
        "--sampling-seed", str(SAMPLING_SEED),
    ]
    if val_csv is not None:
        cmd += ["--val-attention-csv", str(val_csv)]
    print(f"\n=== {label} ===")
    rc = subprocess.run(cmd, text=True).returncode
    print(f"[{'DONE' if rc == 0 else 'FAILED'}] {label} → {out_file}")
    return rc

for cfg in tcga_fold_configs:
    run_gradcam(
        ckpt=cfg["ckpt"],
        features_dir=TCGA_FEATURES_DIR,
        out_file=cfg["out_file"],
        val_csv=cfg["val_csv"],
        label=f"TCGA fold{cfg['fold']}",
    )

## A3. Inspect TCGA Output

In [ ]:
import pandas as pd

dfs = []
for cfg in tcga_fold_configs:
    f = cfg["out_file"]
    if f.exists():
        df = pd.read_parquet(f)
        df["fold"] = f"fold{cfg['fold']}"
        dfs.append(df)
        print(f"fold{cfg['fold']}: {len(df):,} rows — {f.stat().st_size/1e6:.1f} MB")
    else:
        print(f"fold{cfg['fold']}: output not found")

if dfs:
    tcga_full = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal TCGA: {len(tcga_full):,} rows | {tcga_full['feature_file'].nunique()} slides | {tcga_full['signature_name'].nunique()} signatures")
    display(tcga_full.head(3))

---
# Part B — CPTAC (External Test, Refit Model)

CPTAC uses **`hist2sig.pt`**: the model trained on the full TCGA dataset (train + val combined).
No `val_attention_csv` is needed — the feature directory already contains only CPTAC slides.

## B1. CPTAC Prerequisites Check

In [ ]:
cptac_ckpt      = CHECKPOINTS_DIR / "hist2sig.pt"
cptac_out_file  = CPTAC_OUT_BASE / "cptac_gradcam.parquet"

ok = True

if not cptac_ckpt.exists():
    print(f"[MISSING] hist2sig.pt: {cptac_ckpt}"); ok = False
else:
    print(f"[OK] hist2sig.pt: {cptac_ckpt}")

if not CPTAC_FEATURES_DIR.exists():
    print(f"[MISSING] CPTAC features: {CPTAC_FEATURES_DIR}"); ok = False
else:
    n = len(list(CPTAC_FEATURES_DIR.glob("*.pt")))
    print(f"[OK] CPTAC features: {n} .pt files")

print()
print("CPTAC OK" if ok else "Fix missing files before proceeding.")

## B2. Run CPTAC Attribution

In [ ]:
CPTAC_OUT_BASE.mkdir(parents=True, exist_ok=True)

# No val_csv: process all slides in the CPTAC feature directory
run_gradcam(
    ckpt=cptac_ckpt,
    features_dir=CPTAC_FEATURES_DIR,
    out_file=cptac_out_file,
    val_csv=None,
    label="CPTAC refit",
)

## B3. Inspect CPTAC Output

In [ ]:
if cptac_out_file.exists():
    cptac_df = pd.read_parquet(cptac_out_file)
    print(f"CPTAC: {len(cptac_df):,} rows — {cptac_out_file.stat().st_size/1e6:.1f} MB")
    print(f"Slides: {cptac_df['feature_file'].nunique()} | Signatures: {cptac_df['signature_name'].nunique()}")
    display(cptac_df.head(3))
else:
    print(f"Output not found: {cptac_out_file}")

---
# grad_score Distribution (TCGA + CPTAC)

Visual comparison of score distributions across the top 6 signatures.

In [ ]:
import matplotlib.pyplot as plt

datasets = []
if dfs:
    datasets.append(("TCGA", tcga_full))
if cptac_out_file.exists():
    datasets.append(("CPTAC", cptac_df))

if not datasets:
    print("No output available.")
else:
    ref_df   = datasets[0][1]
    top_sigs = ref_df["signature_name"].value_counts().index[:6].tolist()
    colors   = {"TCGA": "steelblue", "CPTAC": "tomato"}

    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    for ax, sig in zip(axes.flat, top_sigs):
        for name, df in datasets:
            vals = df[df["signature_name"] == sig]["grad_score"].clip(-1, 1)
            ax.hist(vals, bins=80, alpha=0.6, color=colors[name], label=name, edgecolor="none")
        ax.axvline(0, color="black", lw=0.8, ls="--")
        ax.set_title(sig, fontsize=9)
        ax.legend(fontsize=7)
    plt.suptitle("grad_score distribution by signature", fontsize=11)
    plt.tight_layout()
    plt.show()